In [1]:
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader

To understand the dataloader, which uses a sliding window approach, we will create a simple dataset of numbers in increasing order:

In [2]:
file_path = Path("number-data.txt")
with open(file_path, "w", encoding="utf-8") as f:
    for number in range(1001):
        f.write(f"{number} ")

We will adjust our dataset class to use integer tokens parsed directly from the text:

In [3]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt: str, max_length:int, stride:int) -> None:
        self.input_ids = []
        self.target_ids = []

        token_ids = [int(i) for i in txt.strip().split()]

        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i: i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))
    
    def __len__(self) -> int:
        return len(self.input_ids)

    def __getitem__(self, index) -> tuple[torch.Tensor, torch.Tensor]:
        return self.input_ids[index], self.target_ids[index]

In [4]:
def create_dataloader_v1(txt:str, batch_size:int=4, max_length:int=256, stride:int=128, shuffle:bool=True, drop_last:bool=True, num_workers:int=0) -> DataLoader:
    dataset = GPTDatasetV1(txt, max_length, stride)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

In [5]:
with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

In [6]:
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[0, 1, 2, 3]]), tensor([[1, 2, 3, 4]])]


In [7]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[1, 2, 3, 4]]), tensor([[2, 3, 4, 5]])]


In [8]:
third_batch = next(data_iter)
print(third_batch)

[tensor([[2, 3, 4, 5]]), tensor([[3, 4, 5, 6]])]


In [9]:
all_batches = list(dataloader)
last_batch = all_batches[-1]
print(last_batch)

[tensor([[996, 997, 998, 999]]), tensor([[ 997,  998,  999, 1000]])]


Batched inputs with a larger stride:

In [10]:
dataloader = create_dataloader_v1(raw_text, batch_size=2, max_length=4, stride=4, shuffle=False)

for inputs, targets in dataloader:
    pass

print(f"Inputs:\n {inputs}")
print(f"Targets:\n {targets}")

Inputs:
 tensor([[992, 993, 994, 995],
        [996, 997, 998, 999]])
Targets:
 tensor([[ 993,  994,  995,  996],
        [ 997,  998,  999, 1000]])


With shuffling:

In [11]:
torch.manual_seed(123)
dataloader = create_dataloader_v1(raw_text, batch_size=2, max_length=4, stride=4, shuffle=True)

for inputs, targets in dataloader:
    pass

print(f"Inputs:\n {inputs}")
print(f"Targets:\n {targets}")

Inputs:
 tensor([[880, 881, 882, 883],
        [112, 113, 114, 115]])
Targets:
 tensor([[881, 882, 883, 884],
        [113, 114, 115, 116]])
